# Oil Spill Detection via SAR
**NB 14** | Persian Gulf | Sentinel-1 GRD | Backscatter threshold + Spatial analysis

In [ ]:
import warnings; warnings.filterwarnings('ignore')
import pathlib, json
import numpy as np
import matplotlib.pyplot as plt
import pygeovision as pgv
from pygeovision.models import get_model
from pygeovision.inference.tiled import TiledInference

client = pgv.PyGeoVision()
print(client)


In [ ]:
BBOX       = (50.50, 26.50, 51.00, 27.00)  # Persian Gulf, offshore Kuwait/Iraq
DATE_RANGE = ('2024-01-01', '2024-12-31')
PROVIDERS  = ['planetary_computer']         # Sentinel-1 GRD available on PC
BANDS      = ['vv','vh']                   # Sentinel-1 dual-pol GRD
DATA_DIR   = pathlib.Path('./results/nb14')
DATA_DIR.mkdir(parents=True, exist_ok=True)


In [ ]:
results = client.search(
    bbox=BBOX, date_range=DATE_RANGE, providers=PROVIDERS,
    satellites=['Sentinel-1'],  # explicitly request SAR
    cloud_cover_max=100,        # SAR is cloud-independent
)
print(f"SAR scenes found: {len(results)}")
for r in results[:5]:
    print(f"  {r.datetime[:10]}  {r.satellite}  {r.provider}")


In [ ]:
downloads = client.download(results[:1], output_dir=str(DATA_DIR),
    bands=BANDS, post_process=['reproject:EPSG:32639','cog'])
scene_path = downloads[0].path if downloads and downloads[0].success else None
print(f"SAR scene: {scene_path}")


In [ ]:
# Validate SAR data — note: SAR has different value range than optical
if scene_path:
    raw_report = client.validator.validate(str(scene_path), required_bands=len(BANDS))
    print("SAR raw validation:")
    print(raw_report.summary())


In [ ]:
# Preprocess SAR: clip to AOI, no SCL (SAR is cloud-free), normalise using minmax
PREPROCESSED = DATA_DIR / 'gulf_sar_preprocessed.tif'
if scene_path:
    ready = client.prepare_for_ai(
        str(scene_path),
        stack_bands = BANDS,
        bbox        = BBOX,
        # No SCL for SAR — cloud masking not needed
        normalise   = 'minmax',          # SAR backscatter to [0,1] per band
        model_type  = 'segmentation',
        output_path = str(PREPROCESSED),
    )
    print("SAR preprocessing steps:", ready['steps'])
    print("Shape (C,H,W)          :", ready['shape'])
    print("Value range [0,1]      :", ready['array'].min().round(4), '→', ready['array'].max().round(4))


In [ ]:
# Oil spill detection: backscatter threshold on VV channel
# Dark spots (low backscatter) in VV = oil slick candidates
import rasterio, numpy as np

SPILL_MAP  = DATA_DIR / 'oil_spill_mask.tif'
PREDICTION = SPILL_MAP

if PREPROCESSED.exists():
    client.postprocess.apply_confidence_threshold(
        str(PREPROCESSED),    # VV normalised — low = potential spill
        str(SPILL_MAP),
        threshold = 0.15,     # below 15% of max backscatter = dark anomaly
        band      = 1,        # VV polarisation
        nodata    = 255,
    )
    # Apply SAR despeckle via bridge (PyGeoFetch v2.0 native when available)
    # SAR despeckle via pygeovision.data.processors.sar.despeckle_sar()
    # (correct order: apply BEFORE oil-spill threshold detection)
    _desp_path = str(DATA_DIR / 'preprocessed_despeckled.tif')
    try:
        despeckle_sar(str(PREPROCESSED), _desp_path, filter_type='enhanced_lee', window_size=5)
        PREPROCESSED = pathlib.Path(_desp_path)   # use despeckled version
    except Exception as _e:
        print(f'Despeckle skipped ({_e}); using pre-despeckled input')
    print(f"Oil spill mask: {SPILL_MAP}")


In [ ]:
if PREDICTION.exists():
    pred_report = client.validator.validate(str(PREDICTION))
    print("Prediction validation:", "PASSED" if pred_report.passed else f"FIXED — {pred_report.errors}")
    print(pred_report.summary())


In [ ]:
SIEVED  = DATA_DIR / 'sieved.tif'
VECTOR  = DATA_DIR / 'output.geojson'
COG_OUT = DATA_DIR / 'output_cog.tif'

if PREDICTION.exists():
    client.postprocess.sieve_filter(str(PREDICTION), min_pixels=8, output_path=str(SIEVED))
    client.postprocess.vectorise(str(SIEVED), str(VECTOR),
        target_class=1, min_area_m2=100.0, simplify_tolerance=1.0)
    stats = client.postprocess.class_statistics(str(SIEVED))
    client.postprocess.to_cog(str(SIEVED), str(COG_OUT))
    with open(VECTOR) as f: gj = json.load(f)
    print(f"Polygons detected : {len(gj['features'])}")
    for cls, info in sorted(stats.items()):
        print(f"  Class {cls}: {info['pixels']:>8,} px | {info['area_ha']:>8.1f} ha | {info['pct']:>6.1f}%")
    print(f"COG saved: {COG_OUT}")


In [ ]:
# Area and statistics for detected spills
stats  = client.postprocess.class_statistics(str(DATA_DIR/'sieved.tif'))
spill_ha = stats.get(1,{}).get('area_ha',0)
print(f"Oil slick candidates : {spill_ha:.1f} ha total")
print(f"Coverage             : {stats.get(1,{}).get('pct',0):.2f}%")


**NB14 — Oil Spill** complete.
- Study area: Persian Gulf
- Sensor: Sentinel-1 GRD SAR
- Model: Backscatter threshold + SAR analysis
- Pipeline: search → download → validate_raw → preprocess → validate_clean → model → postprocess → COG